In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
df_lic = pd.read_parquet("data/data_licences/data_licences.parquet")
df_med = pd.read_csv("data/data_medailles/data_medailles_jo.csv")
df_or = pd.read_csv("data/data_medailles/data_or_jo.csv")

In [ ]:
df_lic.head()

In [ ]:
def plot_licences_par_annee(df_lic, sport_code=None, sport_col="code_sport"):
    """
    Affiche un graphique interactif des licences annuelles.
    
    Paramètres :
    ------------
    df_lic : DataFrame avec colonnes 'annee', 'licences_annuelles' et 'code_sport'
    sport_code : filtrer par code de sport (optionnel)
    sport_col : nom de la colonne contenant le code sport
    """
    
    df = df_lic.copy()
    
    if sport_code is not None:
        df = df[df[sport_col] == sport_code]
    
    data = (
        df.groupby("annee")["licences_annuelles"]
          .sum()
          .reset_index()
          .sort_values("annee")
    )
    
    if len(data) >= 2:
        data["variation"] = data["licences_annuelles"].diff()
        data["taux_evolution_%"] = data["licences_annuelles"].pct_change() * 100
    else:
        data["variation"] = np.nan
        data["taux_evolution_%"] = np.nan
        
    titre = f"Licences annuelles - {sport_col}"
    
    fig = px.line(
        data, x="annee", y="licences_annuelles",
        title=titre, markers=True,
        labels={"annee": "Année", "licences_annuelles": "Licences annuelles"}
    )
    
    fig.update_layout(xaxis=dict(dtick=1))
    fig.show()


In [ ]:
# Code interactif

# --- Préparer les options pour les widgets ---
# Créer le widget
sports = sorted(df_lic["code_sport"].dropna().unique())

# Ajouter l'option "all" pour toutes les tranches
options = ["all"] + list(sports)

sports_widget = widgets.Dropdown(
    options=options,
    description="Sports :",
    value="all"
)
out = widgets.Output()

def update_graph(change=None):
    clear_output(wait=True)
    display(sports_widget)
    plot_licences_par_annee(df_lic, sport_code=sports_widget.value)

# --- Lier les widgets ---
sports_widget.observe(update_graph, names='value')

# --- Affichage initial ---
display(sports_widget, out)
update_graph()


In [ ]:
def plot_licences_par_sexe(df_lic, sport_code=None, sport_col="code_sport"):
    df = df_lic.copy()
    
    if sport_code is not None:
        df = df[df[sport_col] == sport_code]
    
    # Remplacer les valeurs manquantes par 'NR'
    df['sexe'] = df['sexe'].fillna('NR')
    
    # Obtenir toutes les années et toutes les catégories de sexe
    annees = sorted(df['annee'].unique())
    sexes = df['sexe'].unique()
    
    # Créer un DataFrame complet avec toutes les combinaisons année x sexe
    complete = pd.MultiIndex.from_product([annees, sexes], names=['annee', 'sexe']).to_frame(index=False)
    
    # Somme des licences par année et sexe
    data = df.groupby(['annee', 'sexe'])['licences_annuelles'].sum().reset_index()
    
    # Merge avec le DataFrame complet pour avoir des 0 si absence de données
    data = complete.merge(data, on=['annee', 'sexe'], how='left').fillna(0)
    
    # Pivot pour le tracé
    pivot = data.pivot(index='annee', columns='sexe', values='licences_annuelles')
    
    # Création du graphique
    fig = go.Figure()
    for sexe in pivot.columns:
        fig.add_trace(go.Scatter(
            x=pivot.index,
            y=pivot[sexe],
            mode='lines+markers',
            name=str(sexe)
        ))
    
    fig.update_layout(
        title=f"Licences annuelles par sexe - {sport_code}",
        xaxis_title="Année",
        yaxis_title="Licences annuelles",
        xaxis=dict(dtick=1),
        legend_title="Sexe"
    )
    
    fig.show()


In [ ]:
# Code interactif

# --- Préparer les options pour les widgets ---
# Créer le widget
sports = sorted(df_lic["code_sport"].dropna().unique())

# Ajouter l'option "all" pour toutes les tranches
options = ["all"] + list(sports)

sports_widget = widgets.Dropdown(
    options=options,
    description="Sports :",
    value="all"
)
out = widgets.Output()

def update_graph(change=None):
    clear_output(wait=True)
    display(sports_widget)
    plot_licences_par_sexe(df_lic, sport_code=sports_widget.value)

# --- Lier les widgets ---
sports_widget.observe(update_graph, names='value')

# --- Affichage initial ---
display(sports_widget, out)
update_graph()


In [ ]:
#lic en dessous de age_max

def get_licences_jeunes(df_lic, age_max=15, sport_code=None, sport_col="code_sport"):
    df = df_lic.copy()

    if sport_code is not None:
        df = df[df[sport_col] == sport_code]

    total = (
        df.groupby("annee")["licences_annuelles"]
          .sum()
          .reset_index()
          .rename(columns={"licences_annuelles": "licences_total"})
    )

    jeunes = (
        df[df["age"] < age_max]
        .groupby("annee")["licences_annuelles"]
        .sum()
        .reset_index()
        .rename(columns={"licences_annuelles": "licences_jeunes"})
    )

    res = jeunes.merge(total, on="annee", how="left")
    res["part_jeunes_%"] = res["licences_jeunes"] / res["licences_total"] * 100

    return res


In [ ]:
def plot_part_jeunes(df_lic, age_max=15, sport_code=None, sport_col="code_sport"):
    data = get_licences_jeunes(df_lic, age_max, sport_code, sport_col)

    plt.figure(figsize=(9, 5))
    plt.plot(data["annee"], data["part_jeunes_%"], marker="o")
    titre = f"Part des licenciés < {age_max} ans"
    if sport_code is not None:
        titre += f" - sport {sport_code}"
    plt.title(titre)
    plt.xlabel("Année")
    plt.ylabel("Part des jeunes (%)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
df_lic["age"] = pd.to_numeric(df_lic["age"], errors="coerce")

# global tous sports
plot_licences_par_annee(df_lic)
plot_licences_par_sexe(df_lic)
plot_part_jeunes(df_lic, age_max=15)

# uniquement Hand si ton code_sport du hand = "HAND"
plot_licences_par_annee(df_lic, sport_code="HAN")
plot_licences_par_sexe(df_lic, sport_code="HAN")
plot_part_jeunes(df_lic, age_max=15, sport_code="HAN")


In [ ]:
def plot_hand_ratio_f_h(df_lic, sport_code="HAN", sport_col="code_sport"):
    df = df_lic.copy()
    df = df[df[sport_col] == sport_code]

    df_par_sexe = (
        df.groupby(["annee", "sexe"])["licences_annuelles"]
          .sum()
          .reset_index()
          .pivot(index="annee", columns="sexe", values="licences_annuelles")
    )

    df_par_sexe["ratio_f_h"] = df_par_sexe["F"] / df_par_sexe["H"]
    df_par_sexe.head()
    plt.figure(figsize=(10, 5))
    plt.plot(df_par_sexe.index, df_par_sexe["ratio_f_h"], marker="o")
    plt.title("Handball – Ratio Femmes / Hommes")
    plt.xlabel("Année")
    plt.ylabel("Ratio F / H")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
df_lic.head()

In [ ]:
plot_hand_ratio_f_h(df_lic, sport_code="HAN", sport_col="code_sport")

In [ ]:
import seaborn as sns

def plot_hand_heatmap(df_lic, sport_code="HAN", sport_col="code_sport"):
    df = df_lic.copy()
    df = df[df[sport_col] == sport_code]

    pivot = df.groupby(["annee", "tranche_age"])["licences_annuelles"] \
              .sum().reset_index().pivot(index="tranche_age", columns="annee", values="licences_annuelles")

    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot, cmap="Blues", annot=False)
    plt.title("Handball – Heatmap Licences (année × tranche d’âge)")
    plt.tight_layout()
    plt.show()


In [ ]:
plot_hand_heatmap(df_lic, sport_code="HAN", sport_col="code_sport")

In [ ]:
df_lic.columns